# 🔬 Agricultural Pest Image Retrieval System (CBIR) on Kaggle
Notebook này hướng dẫn chi tiết cách thiết lập môi trường, chạy hệ thống truy vấn ảnh sâu bệnh nông nghiệp 2 giai đoạn (**Automatic Detection & Crop -> CLIP Search**) trên tập dữ liệu **IP102** cho cả **4 Tasks** nhằm thu thập chỉ số Recall@1/5/10 cho báo cáo đánh giá.

### ⚠️ Yêu cầu trước khi chạy:
1. Chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** hoặc **GPU P100** trong phần settings của Kaggle (*Accelerator -> GPU*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Thiết lập môi trường và tải repository

In [ ]:
# 1. Clone repository chứa code mới nhất
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# 2. Tải submodule mmyolo
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi phiên bản MMCV

In [ ]:
print("-> 1. Cài đặt các gói PyTorch & Torchvision tương thích cu121...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ pre-built index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV...")
import site
import glob

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        init_file = os.path.join(s_dir, pkg, "__init__.py")
        if os.path.exists(init_file):
            with open(init_file, 'r', encoding='utf-8') as f:
                content = f.read()
            content = content.replace("mmcv_maximum_version = '2.2.0'", "mmcv_maximum_version = '2.3.0'")
            content = content.replace("mmcv_maximum_version = '2.1.0'", "mmcv_maximum_version = '2.3.0'")
            with open(init_file, 'w', encoding='utf-8') as f:
                f.write(content)

print("====== Khởi tạo môi trường thành công! ======")

## 🗂️ Bước 3: Tìm kiếm Dataset & Định nghĩa Checkpoints
Hãy điều chỉnh giá trị của `CHECKPOINT_DIR` ở cell dưới đây tương ứng với đường dẫn chứa các file checkpoint của bạn (`t1_best.pth`, `t2_best.pth`...) trên Kaggle.

In [ ]:
import glob

# Tự động định vị thư mục dataset IP102 trên Kaggle
dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break

if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")

# ĐỊNH NGHĨA ĐƯỜNG DẪN CHECKPOINT CỦA BẠN TRÊN KAGGLE
# Hãy thay đổi giá trị dưới đây tương ứng với Kaggle Input dataset chứa các file checkpoint của bạn
CHECKPOINT_DIR = "/kaggle/input/your-trained-checkpoints-folder"

# Xác nhận sự tồn tại của file checkpoint mẫu để kiểm tra
t1_ckpt = os.path.join(CHECKPOINT_DIR, "t1_best.pth")
if os.path.exists(t1_ckpt):
    print(f"-> Phát hiện checkpoint Task 1 tại: {t1_ckpt}")
else:
    print(f"⚠️ Cảnh báo: Chưa tìm thấy checkpoint tại '{t1_ckpt}'. Vui lòng cập nhật biến CHECKPOINT_DIR.")

## 🔍 Bước 4: Chạy thử tìm kiếm tương đồng trên một ảnh (Query Single Image)
Cell này thực hiện quy trình tìm kiếm ảnh sâu bệnh tương đồng trên một bức ảnh truy vấn bất kỳ và hiển thị kết quả so sánh trực quan.

In [ ]:
from IPython.display import Image, display
import glob

# Cấu hình các tham số chạy truy vấn thử nghiệm
t1_config = "configs/open_world/mowod/custom/ip102_t1.py"
t1_checkpoint = os.path.join(CHECKPOINT_DIR, "t1_best.pth")
gallery_folder = os.path.join(dataset_root, "test")
ann_file = os.path.join(dataset_root, "test.json")

# Tự động tìm kiếm ảnh thực tế bất kỳ trong tập test để truy vấn tránh FileNotFoundError
test_images = glob.glob(os.path.join(dataset_root, "**/*.jpg"), recursive=True)
test_images = [f for f in test_images if "test" in f.replace("\\", "/")]
if test_images:
    query_image_path = test_images[0]
    print(f"-> Tự động tìm thấy ảnh query thực tế: {query_image_path}")
else:
    query_image_path = os.path.join(dataset_root, "test", "00001.jpg")
    print(f"-> Cảnh báo: Không tìm thấy ảnh .jpg. Dùng fallback: {query_image_path}")

# 1. Xây dựng index cho Gallery
print("-> Đang khởi tạo CSDL đặc trưng ảnh mẫu (Gallery Index)... (Có thể mất 2-3 phút)")
!python build_gallery_index.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-dir "{gallery_folder}" \
    --ann-file "{ann_file}" \
    --output gallery_index_task1.pkl

# 2. Tiến hành truy vấn ảnh
print("\n-> Đang chạy truy vấn ảnh sâu bệnh...")
!python retrieve.py \
    --config "{t1_config}" \
    --checkpoint "{t1_checkpoint}" \
    --gallery-index gallery_index_task1.pkl \
    --query-image "{query_image_path}" \
    --top-k 5 \
    --anomaly-thr 0.55 \
    --output query_result.jpg

# 3. Hiển thị kết quả tìm kiếm trực quan trực tiếp trong notebook
if os.path.exists("query_result.jpg"):
    display(Image(filename="query_result.jpg"))
else:
    print("Error: Không tìm thấy ảnh kết quả 'query_result.jpg'.")

## 📈 Bước 5: Chạy đánh giá đo chỉ số Recall@1/5/10 cho cả 4 Tasks
Cell này chạy đánh giá tuần tự cho cả 4 task và xuất báo cáo điểm số chi tiết từng loài sâu bệnh ra các file markdown.

In [ ]:
tasks = [
    {"id": 1, "config": "configs/open_world/mowod/custom/ip102_t1.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "t1_best.pth")},
    {"id": 2, "config": "configs/open_world/mowod/custom/ip102_t2.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "t2_best.pth")},
    {"id": 3, "config": "configs/open_world/mowod/custom/ip102_t3.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "t3_best.pth")},
    {"id": 4, "config": "configs/open_world/mowod/custom/ip102_t4.py", "checkpoint": os.path.join(CHECKPOINT_DIR, "t4_best.pth")}
,]

for t in tasks:
    task_id = t["id"]
    config_path = t["config"]
    ckpt_path = t["checkpoint"]
    
    print("\n" + "="*60)
    print(f"   ĐANG CHẠY ĐÁNH GIÁ THỰC NGHIỆM CHO TASK {task_id}   ")
    print("="*60)
    
    if not os.path.exists(ckpt_path):
        print(f"⚠️ Bỏ qua Task {task_id} vì không tìm thấy file checkpoint tại: {ckpt_path}")
        continue
        
    # Khởi chạy script đánh giá
    !python evaluate_retrieval.py \
        --config "{config_path}" \
        --checkpoint "{ckpt_path}" \
        --dataset-root "{dataset_root}" \
        --query-split val \
        --gallery-split test \
        --query-cache "query_cache_task{task_id}.pkl" \
        --gallery-cache "gallery_cache_task{task_id}.pkl" \
        --output-report "report_task{task_id}.md"

## 📄 Bước 6: Đọc kết quả các Task phục vụ báo cáo

In [ ]:
from IPython.display import Markdown, display

for i in [1, 2, 3, 4]:
    report_file = f"report_task{i}.md"
    if os.path.exists(report_file):
        print(f"\n\n🔍 KẾT QUẢ ĐÁNH GIÁ CHỈ SỐ TASK {i} (Được đọc từ {report_file}):")
        display(Markdown(filename=report_file))
    else:
        print(f"-> Không tìm thấy báo cáo kết quả của Task {i} (Chưa chạy đánh giá hoặc lỗi file).")